### This notebook is an exploration of a single QM hydroxide molecule interacting with a single MM water. 

The hydroxide is defined quantum-mechanically. In PySCF, its ab initio calculation is called an RHF calculation. It's done via a call to pyscf.gto.M, which creates a pyscf "Mole" object named "mol" using coordinates provided in that call. Those coordinates are kept in numpy array qm_coords. This is an electronic SCF optimization only -- PySCF keeps the nuclei where specified. 

The water is defined with molecular mechanics, in two ways:

1. As *charges*. These charges are kept in mm_charges, positive for hydrogens and negative for the oxygens. Coordinates of the charges are kept in array mm_charge_coords.
1. As *6-12 Lennard-Jones objects*. In this case it's LJ for the oxygen only.

There are two RHF calculations done here. One is the bare hydroxide, the other is the hydroxide that knows about the charges on the water. Each produces an energy.

There's also an energy calculation that include the effect of LJ interactions between hydroxide and water, but the LJ calculation does not feed back into the RHF calculation. The electronic density in the second RHF calculation responds to the MM charges, but not to the Lennard-Jones potential. The LJ energy is added afterward as a classical correction.

In [10]:
import numpy as np
from pyscf import gto, scf, qmmm

In [11]:
# ============================================================
# Constants
# ============================================================

HARTREE_TO_KJMOL = 2625.49962

In [12]:
# ============================================================
# Lennard-Jones energy
# ============================================================

def lj_energy(
    qm_coords,
    qm_atom_types,
    mm_coords,
    mm_atom_types,
    lj_params
):
    """
    Calculate QM-MM Lennard-Jones energy.

    Parameters
    ----------
    qm_coords : (N,3) array
        QM coordinates in Angstrom.

    qm_atom_types : list
        LJ atom type for each QM atom.

    mm_coords : (M,3) array
        MM LJ-site coordinates in Angstrom.

    mm_atom_types : list
        LJ atom type for each MM particle.

    lj_params : dict
        Dictionary containing sigma (Angstrom) and
        epsilon (kJ/mol) for each atom type.

    Returns
    -------
    total : float
        Total QM-MM LJ energy in kJ/mol.
    """

    total = 0.0

    print()
    print("Individual QM-MM LJ interactions")
    print("----------------------------------------")

    for i, r_qm in enumerate(qm_coords):

        for j, r_mm in enumerate(mm_coords):

            qm_type = qm_atom_types[i]
            mm_type = mm_atom_types[j]

            r = np.linalg.norm(r_qm - r_mm)

            if r == 0.0:
                raise ValueError(
                    f"Zero distance between QM atom {i} "
                    f"and MM atom {j}"
                )

            sigma_qm = lj_params[qm_type]["sigma_A"]
            epsilon_qm = lj_params[qm_type]["epsilon_kJmol"]

            sigma_mm = lj_params[mm_type]["sigma_A"]
            epsilon_mm = lj_params[mm_type]["epsilon_kJmol"]

            # Lorentz-Berthelot mixing rules
            sigma = 0.5 * (sigma_qm + sigma_mm)
            epsilon = np.sqrt(epsilon_qm * epsilon_mm)

            sr6 = (sigma / r) ** 6
            sr12 = sr6 ** 2

            e = 4.0 * epsilon * (sr12 - sr6)

            total += e

            print(
                f"QM {i:2d} ({qm_type:6s})  "
                f"MM {j:2d} ({mm_type:6s})  "
                f"r = {r:8.4f} Å   "
                f"E = {e:14.6f} kJ/mol"
            )

    return total

In [13]:
# ============================================================
# 1. QM SYSTEM
#
# OH-
#
# O-H distance = 0.97 Å
#
# Charge = -1
# ============================================================

mol = gto.M(
    atom='''
        O    0.000    0.000    0.000
        H    0.970    0.000    0.000
    ''',
    basis='sto-3g',
    charge=-1,
    spin=0,
    unit='Angstrom',
    verbose=4
)

System: uname_result(system='Linux', node='Cas2', release='6.1.177-1-MANJARO', version='#1 SMP PREEMPT_DYNAMIC Sat Jul  4 22:18:29 UTC 2026', machine='x86_64')  Threads 32
Python 3.14.6 (main, Jun 15 2026, 11:36:54) [GCC 16.1.1 20260430]
numpy 2.5.1  scipy 1.18.0  h5py 3.16.0
Date: Mon Aug 24 07:59:04 2026
PySCF version 2.14.0
PySCF path  /home/chemistry/venvs/jupyter/lib/python3.14/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 10
[INPUT] charge = -1
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = Angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O      0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0
[INPUT]  2 H      0.970000000000   0.000000000000   0.000000000000 AA    1.833034340828   0.000000000

In [14]:
# ============================================================
# 2. TIP4P-D WATER
#
# We use the TIP4P-D electrostatic sites:
#
# H1 = +0.58
# H2 = +0.58
# M  = -1.16
#
# The oxygen itself has zero charge in the TIP4P
# electrostatic model.
#
# Water is placed approximately 3 Å away from OH-.
# ============================================================

water_O = np.array([3.5, 0.0, 0.0])

water_H1 = water_O + np.array([
    0.757,
    0.586,
    0.000
])

water_H2 = water_O + np.array([
   -0.757,
    0.586,
    0.000
])

water_M = water_O + np.array([
    0.000,
    0.151,
    0.000
])

In [16]:
# ============================================================
# Electrostatic sites
# ============================================================

mm_charge_coords = np.array([
    water_H1,
    water_H2,
    water_M
])

mm_charges = np.array([
    +0.58,
    +0.58,
    -1.16
])

print('water charge coordinates are \n', mm_charge_coords)
print('water charges re \n', mm_charges)

water charge coordinates are 
 [[4.257 0.586 0.   ]
 [2.743 0.586 0.   ]
 [3.5   0.151 0.   ]]
water charges re 
 [ 0.58  0.58 -1.16]


In [8]:
# ============================================================
# LJ sites
#
# For LJ, TIP4P water has its LJ interaction on oxygen.
#
# Therefore:
#
#   MM LJ sites = O only
#
# The H and M sites participate in electrostatics but
# NOT in the LJ calculation.
# ============================================================

mm_lj_coords = np.array([
    water_O
])

mm_lj_types = [
    "TIP4P_O"
]

print('water Lennard-Jones coordinates are ', mm_lj_coords)

water Lennard-Jones coordinates are  [[3.5 0.  0. ]]


In [8]:
# ============================================================
# 3. BARE QM CALCULATION
# ============================================================

mf_qm = scf.RHF(mol)

energy_qm_hartree = mf_qm.kernel()

energy_qm_kjmol = (
    energy_qm_hartree * HARTREE_TO_KJMOL
)



******** <class 'pyscf.scf.hf.RHF'> ********
method = RHF
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = /tmp/tmpagnr4d3c
max_memory 4000 MB (current use 175 MB)
Set gradient conv threshold to 3.16228e-05
Initial guess from minao.
init E= -73.9765896919984
  HOMO = -0.336126307462797  LUMO = 0.4885099519304  gap/eV = 22.43950
cycle= 1 E= -73.9300784358367  delta_E= 0.0465  |g|= 0.529  |ddm|=  1.5
  HOMO = 0.4145410829168  LUMO = 1.24739187792404  gap/eV = 22.66302
cycle= 2 E= -74.0560473499752  delta_E= -0.126  |g|= 0.0535  |ddm|= 0.869
  HOMO = 0.251789068399814  LUMO = 1.24576848161727  gap/eV = 27.04756
cycle= 3 E= -74.0573907484135  delta_E= -0.00134  |g|= 0.00458  |ddm|= 0.11
  HOMO = 0.251112681742943  LUMO = 1.24325478899863  gap/eV = 26.99

In [9]:
# ============================================================
# 4. QM/MM ELECTROSTATIC EMBEDDING
#
# This is the actual PySCF QMMM calculation.
#
# The TIP4P charges are incorporated into the QM
# Hamiltonian, so the OH- electron density responds
# self-consistently to the water electrostatic field.
# ============================================================

mf_qmmm = qmmm.mm_charge(
    scf.RHF(mol),
    mm_charge_coords,
    mm_charges,
    unit='Angstrom'
)

energy_qmmm_hartree = mf_qmmm.kernel()

energy_qmmm_kjmol = (
    energy_qmmm_hartree * HARTREE_TO_KJMOL
)



******** <class 'pyscf.qmmm.itrf.QMMMRHF'> ********
method = QMMMRHF
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = /tmp/tmpgyojgior
max_memory 4000 MB (current use 253 MB)
** Add background charges for QMMMRHF **
Set gradient conv threshold to 3.16228e-05
Initial guess from minao.
init E= -73.9759575537626
  HOMO = -0.34146249221366  LUMO = 0.475321145975872  gap/eV = 22.22581
cycle= 1 E= -73.9356162027169  delta_E= 0.0403  |g|= 0.53  |ddm|=  1.5
  HOMO = 0.406766432804604  LUMO = 1.23655051863289  gap/eV = 22.57958
cycle= 2 E= -74.0621632398274  delta_E= -0.127  |g|= 0.0536  |ddm|= 0.873
  HOMO = 0.243560961836052  LUMO = 1.23435965731849  gap/eV = 26.96101
cycle= 3 E= -74.0635127874064  delta_E= -0.00135  |g|= 0.00458  |ddm|= 0.111
  HOMO = 0.2

In [10]:
# ============================================================
# 5. ELECTROSTATIC QM/MM INTERACTION ENERGY
# ============================================================

energy_electrostatic_kjmol = (
    energy_qmmm_kjmol
    - energy_qm_kjmol
)

In [11]:
# ============================================================
# 6. QM-MM LJ PARAMETERS
#
# These are TEST PARAMETERS.
#
# We can replace these with your actual GROMACS
# parameters once this test is working.
# ============================================================

lj_params = {

    # OH- oxygen
    "OH_O": {
        "sigma_A": 3.400,
        "epsilon_kJmol": 0.2508914038369354
    },

    # OH- hydrogen
    "OH_H": {
        "sigma_A": 1.443,
        "epsilon_kJmol": 0.18390926154006562
    },

    # TIP4P-D oxygen
    #
    # Replace this with the value from your
    # actual TIP4P-D force field.
    "TIP4P_O": {
        "sigma_A": 3.15365,
        "epsilon_kJmol": 0.650194
    }
}


# ============================================================
# QM LJ atom types
# ============================================================

qm_coords = mol.atom_coords(unit='Angstrom')

qm_atom_types = [
    "OH_O",
    "OH_H"
]

In [12]:
# ============================================================
# 7. CALCULATE QM-MM LJ
# ============================================================

energy_lj_kjmol = lj_energy(
    qm_coords,
    qm_atom_types,
    mm_lj_coords,
    mm_lj_types,
    lj_params
)


Individual QM-MM LJ interactions
----------------------------------------
QM  0 (OH_O  )  MM  0 (TIP4P_O)  r =   3.5000 Å   E =      -0.355282 kJ/mol
QM  1 (OH_H  )  MM  0 (TIP4P_O)  r =   2.5300 Å   E =      -0.340480 kJ/mol


In [13]:
# ============================================================
# 8. TOTAL QM/MM INTERACTION
# ============================================================

energy_total_interaction_kjmol = (
    energy_electrostatic_kjmol
    + energy_lj_kjmol
)

In [14]:
# ============================================================
# 9. PRINT RESULTS
# ============================================================

print()
print("======================================================")
print("              QM/MM + LJ TEST")
print("======================================================")

print()
print("Bare QM energy:")
print(
    f"    {energy_qm_hartree:.12f} Hartree"
)
print(
    f"    {energy_qm_kjmol:.6f} kJ/mol"
)

print()
print("QM/MM electrostatic energy:")
print(
    f"    {energy_qmmm_hartree:.12f} Hartree"
)
print(
    f"    {energy_qmmm_kjmol:.6f} kJ/mol"
)

print()
print("QM/MM electrostatic interaction:")
print(
    f"    {energy_electrostatic_kjmol:.6f} kJ/mol"
)

print()
print("QM-MM Lennard-Jones interaction:")
print(
    f"    {energy_lj_kjmol:.6f} kJ/mol"
)

print()
print("------------------------------------------------------")

print("TOTAL QM/MM interaction:")
print(
    f"    {energy_total_interaction_kjmol:.6f} kJ/mol"
)

print("======================================================")


              QM/MM + LJ TEST

Bare QM energy:
    -74.057399189198 Hartree
    -194437.673429 kJ/mol

QM/MM electrostatic energy:
    -74.063521262148 Hartree
    -194453.746930 kJ/mol

QM/MM electrostatic interaction:
    -16.073500 kJ/mol

QM-MM Lennard-Jones interaction:
    -0.695762 kJ/mol

------------------------------------------------------
TOTAL QM/MM interaction:
    -16.769262 kJ/mol
